#Silver Layer Reseller

In [0]:
import pyspark.sql.functions as F
#all necessary libraries are imported

#simple dataframe read to see the bronze delta table
df_reseller = spark.table("accenture_final_project.bronze_layer.reseller")
display(df_reseller)

###Checking the distinct values of the string columns


In [0]:
display(df_reseller.select("Business_Type").distinct())

In [0]:
display(df_reseller.select("Reseller").distinct().orderBy(F.col("Reseller")))

In [0]:
display(df_reseller.select("City").distinct())

In [0]:
display(df_reseller.select("State-Province").distinct())

In [0]:
display(df_reseller.select("Country-Region").distinct())

After examining the data columns we do not find anything to rewrite or delete.

In [0]:
# Calculating nulls for each column
null_counts = df_reseller.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_reseller.columns
])

print("Null count per column:")
display(null_counts)

As we can see we have 0 null values overall so we do not need check farther or delete any rows.

##Cleaning the data

In [0]:
df_reseller_silver = (
    df_reseller.dropDuplicates(["ResellerKey"])
    #1.removing whitespace
    .withColumn("Business_Type", F.trim(F.col("Business_Type")))
    .withColumn("Reseller", F.trim(F.col("Reseller")))
    .withColumn("City", F.trim(F.col("City")))
    .withColumn("State-Province", F.trim(F.col("State-Province")))
    .withColumn("Country-Region", F.trim(F.col("Country-Region")))
    # 2. Renaming columns
    .withColumnRenamed("Business_Type", "BusinessType")
    .withColumnRenamed("State-Province", "StateProvince")
    .withColumnRenamed("Country-Region", "CountryRegion")

).orderBy("ResellerKey")
# Εμφάνιση του αποτελέσματος στο Databricks
display(df_reseller_silver)

##Save the data into a Delta table

In [0]:

(
    df_reseller_silver.write.mode("overwrite")
                            .option("overwriteSchema", "true")
                            .format("delta")
                            .saveAsTable("accenture_final_project.silver_layer.reseller")
)